<a href="https://colab.research.google.com/github/spdb9876/loan-default-prediction/blob/main/01_data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================================
# Home Credit Default Risk — Full Preprocessing Pipeline
# Fixed: chained assignment warnings, DAYS_EMPLOYED logic,
#        AMT_INCOME_TOTAL double-cap, train/test consistency
# ============================================================


In [39]:
import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [40]:

# ── 1. Kaggle setup (Colab) ──────────────────────────────────
from google.colab import userdata

kaggle_json_content = userdata.get('KAGGLE_JSON')
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_json_content)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle API key setup complete.")


Kaggle API key setup complete.


In [41]:

# ── 2. Download & load training data ────────────────────────
competition_name = "home-credit-default-risk"
output_directory = "."

file_name = "application_train.csv"
print(f"Downloading {file_name}...")
os.system(f"kaggle competitions download -c {competition_name} -f {file_name} -p {output_directory}")

zip_path = f"./{file_name}.zip"
csv_path = os.path.join(output_directory, file_name)

if os.path.exists(zip_path) and not os.path.exists(csv_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(output_directory)

df = pd.read_csv(csv_path)
print(f"Loaded training data: {df.shape}")
display(df.head())

Loaded training data: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
# ── 3. Missing value analysis ────────────────────────────────
missing_data       = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100

missing_info = pd.DataFrame({
    'Missing Count':      missing_data,
    'Missing Percentage': missing_percentage
})
missing_info = (
    missing_info[missing_info['Missing Count'] > 0]
    .sort_values('Missing Percentage', ascending=False)
)
print("\nMissing Data Analysis:")
display(missing_info)


Missing Data Analysis:


,Missing Count,Missing Percentage
COMMONAREA_MEDI,214865,69.872297
COMMONAREA_MODE,214865,69.872297
COMMONAREA_AVG,214865,69.872297
NONLIVINGAPARTMENTS_MODE,213514,69.432963
NONLIVINGAPARTMENTS_MEDI,213514,69.432963
...,...,...
EXT_SOURCE_2,660,0.214626
AMT_GOODS_PRICE,278,0.090403
AMT_ANNUITY,12,0.003902
CNT_FAM_MEMBERS,2,0.000650


In [43]:
# ── 4. Drop high-missing columns (> 60 %) ───────────────────
threshold       = 60
columns_to_drop = missing_info[missing_info['Missing Percentage'] > threshold].index.tolist()
print(f"\nDropping {len(columns_to_drop)} columns: {columns_to_drop}")

df_cleaned = df.drop(columns=columns_to_drop)
print(f"Shape after drop: {df_cleaned.shape}")



Dropping 17 columns: ['COMMONAREA_MEDI', 'COMMONAREA_MODE', 'COMMONAREA_AVG', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_AVG', 'FONDKAPREMONT_MODE', 'LIVINGAPARTMENTS_AVG', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAPARTMENTS_MODE', 'FLOORSMIN_MEDI', 'FLOORSMIN_MODE', 'FLOORSMIN_AVG', 'YEARS_BUILD_MODE', 'YEARS_BUILD_MEDI', 'YEARS_BUILD_AVG', 'OWN_CAR_AGE']
Shape after drop: (307511, 105)


In [45]:
# ── 5. Impute remaining missing values ──────────────────────
missing_data_cleaned   = df_cleaned.isnull().sum()
missing_pct_cleaned    = (missing_data_cleaned / len(df_cleaned)) * 100
missing_info_cleaned   = pd.DataFrame({
    'Missing Count':      missing_data_cleaned,
    'Missing Percentage': missing_pct_cleaned
})
missing_info_cleaned = (
    missing_info_cleaned[missing_info_cleaned['Missing Count'] > 0]
    .sort_values('Missing Percentage', ascending=False)
)
print("\nMissing Data after dropping high-missing columns:")
display(missing_info_cleaned)

# Numerical → median  (FIXED: direct assignment, no inplace on copy)
numerical_cols_with_missing = (
    df_cleaned[missing_info_cleaned.index]
    .select_dtypes(include=['int64', 'float64'])
    .columns
)
for col in numerical_cols_with_missing:
    if df_cleaned[col].isnull().any():
        median_val = df_cleaned[col].median()
        df_cleaned[col] = df_cleaned[col].fillna(median_val)
        print(f"  Imputed '{col}' with median {median_val:.4f}")

# Categorical → mode  (FIXED: direct assignment, no inplace on copy)
categorical_cols_with_missing = (
    df_cleaned[missing_info_cleaned.index]
    .select_dtypes(include=['object', 'category'])
    .columns
)
for col in categorical_cols_with_missing:
    if df_cleaned[col].isnull().any():
        mode_val = df_cleaned[col].mode()[0]
        df_cleaned[col] = df_cleaned[col].fillna(mode_val)
        print(f"  Imputed '{col}' with mode '{mode_val}'")

print(f"\nMissing values remaining: {df_cleaned.isnull().sum().sum()}")



Missing Data after dropping high-missing columns:


,Missing Count,Missing Percentage
LANDAREA_AVG,182590,59.376738
LANDAREA_MODE,182590,59.376738
LANDAREA_MEDI,182590,59.376738
BASEMENTAREA_MEDI,179943,58.515956
BASEMENTAREA_MODE,179943,58.515956
BASEMENTAREA_AVG,179943,58.515956
EXT_SOURCE_1,173378,56.381073
NONLIVINGAREA_AVG,169682,55.179164
NONLIVINGAREA_MEDI,169682,55.179164
NONLIVINGAREA_MODE,169682,55.179164


  Imputed 'LANDAREA_AVG' with median 0.0481
  Imputed 'LANDAREA_MODE' with median 0.0458
  Imputed 'LANDAREA_MEDI' with median 0.0487
  Imputed 'BASEMENTAREA_MEDI' with median 0.0758
  Imputed 'BASEMENTAREA_MODE' with median 0.0746
  Imputed 'BASEMENTAREA_AVG' with median 0.0763
  Imputed 'EXT_SOURCE_1' with median 0.5060
  Imputed 'NONLIVINGAREA_AVG' with median 0.0036
  Imputed 'NONLIVINGAREA_MEDI' with median 0.0031
  Imputed 'NONLIVINGAREA_MODE' with median 0.0011
  Imputed 'ELEVATORS_MEDI' with median 0.0000
  Imputed 'ELEVATORS_MODE' with median 0.0000
  Imputed 'ELEVATORS_AVG' with median 0.0000
  Imputed 'APARTMENTS_AVG' with median 0.0876
  Imputed 'APARTMENTS_MODE' with median 0.0840
  Imputed 'APARTMENTS_MEDI' with median 0.0864
  Imputed 'ENTRANCES_AVG' with median 0.1379
  Imputed 'ENTRANCES_MEDI' with median 0.1379
  Imputed 'ENTRANCES_MODE' with median 0.1379
  Imputed 'LIVINGAREA_MODE' with median 0.0731
  Imputed 'LIVINGAREA_AVG' with median 0.0745
  Imputed 'LIVINGARE

In [46]:
# ── 6. Anomaly handling & outlier capping (training) ────────
print("\n--- Outlier Handling ---")

# DAYS_EMPLOYED: replace 365243 with NaN, then impute with median ONCE
anomaly_count = (df_cleaned['DAYS_EMPLOYED'] == 365243).sum()
print(f"DAYS_EMPLOYED anomalies (365243): {anomaly_count}")
if anomaly_count > 0:
    df_cleaned['DAYS_EMPLOYED'] = df_cleaned['DAYS_EMPLOYED'].replace(365243, np.nan)
    days_employed_median = df_cleaned['DAYS_EMPLOYED'].median()
    df_cleaned['DAYS_EMPLOYED'] = df_cleaned['DAYS_EMPLOYED'].fillna(days_employed_median)
    print(f"  Replaced anomalies and re-imputed with median {days_employed_median:.1f}")

# AMT_INCOME_TOTAL: hard cap at 1 000 000
df_cleaned['AMT_INCOME_TOTAL'] = df_cleaned['AMT_INCOME_TOTAL'].clip(lower=0, upper=1e6)
print("  Capped AMT_INCOME_TOTAL at 1 000 000")

# General IQR capping — skip identifier, target and already-handled cols
numerical_cols = df_cleaned.select_dtypes(include=['int64', 'float64']).columns
skip_cols      = {'SK_ID_CURR', 'TARGET', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL'}
cap_cols       = [c for c in numerical_cols if c not in skip_cols]

print(f"  Applying IQR capping to {len(cap_cols)} columns...")
for col in cap_cols:
    Q1  = df_cleaned[col].quantile(0.01)
    Q3  = df_cleaned[col].quantile(0.99)
    IQR = Q3 - Q1
    df_cleaned[col] = df_cleaned[col].clip(
        lower=Q1 - 1.5 * IQR,
        upper=Q3 + 1.5 * IQR
    )
print("Outlier capping complete.")


--- Outlier Handling ---
DAYS_EMPLOYED anomalies (365243): 55374
  Replaced anomalies and re-imputed with median -1648.0
  Capped AMT_INCOME_TOTAL at 1 000 000
  Applying IQR capping to 86 columns...
Outlier capping complete.


In [47]:
# ── 7. Feature engineering ───────────────────────────────────
df_cleaned['CHILDREN_RATIO']      = df_cleaned['CNT_CHILDREN']  / df_cleaned['CNT_FAM_MEMBERS']
df_cleaned['ANNUITY_INCOME_RATIO'] = df_cleaned['AMT_ANNUITY']   / df_cleaned['AMT_INCOME_TOTAL']
df_cleaned['CREDIT_INCOME_RATIO']  = df_cleaned['AMT_CREDIT']    / df_cleaned['AMT_INCOME_TOTAL']
df_cleaned['CREDIT_GOODS_RATIO']   = df_cleaned['AMT_CREDIT']    / df_cleaned['AMT_GOODS_PRICE']

print("New features added:")
display(df_cleaned[['CHILDREN_RATIO','ANNUITY_INCOME_RATIO',
                     'CREDIT_INCOME_RATIO','CREDIT_GOODS_RATIO']].head())

New features added:


/tmp/ipykernel_1700/2321671116.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_cleaned['CREDIT_GOODS_RATIO']   = df_cleaned['AMT_CREDIT']    / df_cleaned['AMT_GOODS_PRICE']


,CHILDREN_RATIO,ANNUITY_INCOME_RATIO,CREDIT_INCOME_RATIO,CREDIT_GOODS_RATIO
0,0.0,0.121978,2.007889,1.158397
1,0.0,0.132217,4.790750,1.145199
2,0.0,0.100000,2.000000,1.000000
3,0.0,0.219900,2.316167,1.052803
4,0.0,0.179963,4.222222,1.000000


In [48]:
# ── 8. Encoding & scaling (training) ────────────────────────
X = df_cleaned.drop('TARGET', axis=1, errors='ignore')
y = df_cleaned['TARGET'] if 'TARGET' in df_cleaned.columns else None

numerical_cols   = [c for c in X.select_dtypes(include=np.number).columns
                    if c != 'SK_ID_CURR']
categorical_cols = X.select_dtypes(include='object').columns.tolist()

print(f"Numerical cols: {numerical_cols[:5]} ...")
print(f"Categorical cols: {categorical_cols[:5]} ...")

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('scaler', StandardScaler())]),        numerical_cols),
    ('cat', Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])

X_processed = preprocessor.fit_transform(X)

encoded_feature_names = (
    preprocessor.named_transformers_['cat']['onehot']
    .get_feature_names_out(categorical_cols)
)
new_column_names = numerical_cols + list(encoded_feature_names)

X_processed_df = pd.DataFrame(
    X_processed if isinstance(X_processed, np.ndarray) else X_processed.toarray(),
    columns=new_column_names,
    index=X.index,
)

print(f"\nProcessed training data shape: {X_processed_df.shape}")
display(X_processed_df.head())

Numerical cols: ['CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE'] ...
Categorical cols: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE'] ...

Processed training data shape: (307511, 228)


,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,...,HOUSETYPE_MODE_terraced house,WALLSMATERIAL_MODE_Block,WALLSMATERIAL_MODE_Mixed,WALLSMATERIAL_MODE_Monolithic,WALLSMATERIAL_MODE_Others,WALLSMATERIAL_MODE_Panel,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,EMERGENCYSTATE_MODE_No,EMERGENCYSTATE_MODE_Yes
0,-0.579728,0.373854,-0.478095,-0.166604,-0.507236,-0.149452,1.506880,0.755835,0.379837,0.579154,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,-0.579728,1.102309,1.725450,0.596658,1.600873,-1.252750,-0.166821,0.497899,1.078697,1.790855,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.579728,-1.083055,-1.152888,-1.412369,-1.092145,-0.783451,-0.689509,0.948701,0.206116,0.306869,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,-0.579728,-0.354601,-0.711430,0.179425,-0.653463,-0.928991,-0.680114,-0.368597,-1.375829,0.369143,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,-0.579728,-0.500292,-0.213734,-0.363353,-0.068554,0.563570,-0.892535,-0.368129,0.191639,-0.307263,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [ ]:
# ── 9. Download & load test data ─────────────────────────────
file_name_test = "application_test.csv"
print(f"\nDownloading {file_name_test}...")
os.system(f"kaggle competitions download -c {competition_name} -f {file_name_test} -p {output_directory}")

zip_path_test = f"./{file_name_test}.zip"
csv_path_test = os.path.join(output_directory, file_name_test)

if os.path.exists(zip_path_test) and not os.path.exists(csv_path_test):
    with zipfile.ZipFile(zip_path_test, 'r') as z:
        z.extractall(output_directory)

df_test = pd.read_csv(csv_path_test)
print(f"Loaded test data: {df_test.shape}")
display(df_test.head())


# ── 10. Preprocess test data ─────────────────────────────────
print("\nPreprocessing test data...")
df_test_cleaned = df_test.copy()

# Drop same columns as training
columns_to_drop_test = [c for c in columns_to_drop if c in df_test_cleaned.columns]
df_test_cleaned = df_test_cleaned.drop(columns=columns_to_drop_test)
print(f"Shape after drop: {df_test_cleaned.shape}")

# DAYS_EMPLOYED anomaly  (FIXED: use training median, single impute, no inplace)
anomaly_count_test = (df_test_cleaned['DAYS_EMPLOYED'] == 365243).sum()
print(f"DAYS_EMPLOYED anomalies in test: {anomaly_count_test}")
if anomaly_count_test > 0:
    df_test_cleaned['DAYS_EMPLOYED'] = df_test_cleaned['DAYS_EMPLOYED'].replace(365243, np.nan)

# Feature engineering (must come before imputation so ratio cols exist)
df_test_cleaned['CHILDREN_RATIO']      = df_test_cleaned['CNT_CHILDREN']  / df_test_cleaned['CNT_FAM_MEMBERS']
df_test_cleaned['ANNUITY_INCOME_RATIO'] = df_test_cleaned['AMT_ANNUITY']   / df_test_cleaned['AMT_INCOME_TOTAL']
df_test_cleaned['CREDIT_INCOME_RATIO']  = df_test_cleaned['AMT_CREDIT']    / df_test_cleaned['AMT_INCOME_TOTAL']
df_test_cleaned['CREDIT_GOODS_RATIO']   = df_test_cleaned['AMT_CREDIT']    / df_test_cleaned['AMT_GOODS_PRICE']
print("New features added to test set.")

# Numerical imputation using training medians  (FIXED: direct assignment)
numerical_cols_test = df_test_cleaned.select_dtypes(include=np.number).columns
for col in numerical_cols_test:
    if df_test_cleaned[col].isnull().any():
        if col == 'DAYS_EMPLOYED':
            median_val = days_employed_median          # training median computed in step 6
        elif col in df_cleaned.columns:
            median_val = df_cleaned[col].median()
        else:
            median_val = df_test_cleaned[col].median()
        df_test_cleaned[col] = df_test_cleaned[col].fillna(median_val)

# Categorical imputation using training modes  (FIXED: direct assignment)
categorical_cols_test = df_test_cleaned.select_dtypes(include='object').columns
for col in categorical_cols_test:
    if df_test_cleaned[col].isnull().any():
        mode_val = (
            df_cleaned[col].mode()[0] if col in df_cleaned.columns
            else df_test_cleaned[col].mode()[0]
        )
        df_test_cleaned[col] = df_test_cleaned[col].fillna(mode_val)

# AMT_INCOME_TOTAL hard cap  (FIXED: unified with training, before general loop)
df_test_cleaned['AMT_INCOME_TOTAL'] = df_test_cleaned['AMT_INCOME_TOTAL'].clip(lower=0, upper=1e6)

# General IQR capping  (FIXED: skip same cols as training, use training bounds)
skip_cols_test        = {'SK_ID_CURR', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL'}
numerical_cols_cap    = [c for c in df_test_cleaned.select_dtypes(include=np.number).columns
                         if c not in skip_cols_test]

for col in numerical_cols_cap:
    if col in df_cleaned.columns:
        Q1  = df_cleaned[col].quantile(0.01)
        Q3  = df_cleaned[col].quantile(0.99)
    else:
        Q1  = df_test_cleaned[col].quantile(0.01)
        Q3  = df_test_cleaned[col].quantile(0.99)
    IQR = Q3 - Q1
    df_test_cleaned[col] = df_test_cleaned[col].clip(
        lower=Q1 - 1.5 * IQR,
        upper=Q3 + 1.5 * IQR
    )

print(f"Missing values remaining in test: {df_test_cleaned.isnull().sum().sum()}")
display(df_test_cleaned.head())
print(f"Test cleaned shape: {df_test_cleaned.shape}")


# ── 11. Apply fitted preprocessor to test data ───────────────
sk_id_curr_test        = df_test_cleaned['SK_ID_CURR']
features_for_proc_test = numerical_cols + categorical_cols
X_test                 = df_test_cleaned[features_for_proc_test]

print(f"X_test shape before transform: {X_test.shape}")
X_test_processed = preprocessor.transform(X_test)

X_test_processed_df = pd.DataFrame(
    X_test_processed if isinstance(X_test_processed, np.ndarray) else X_test_processed.toarray(),
    columns=new_column_names,
    index=X_test.index,
)
X_test_processed_df.insert(0, 'SK_ID_CURR', sk_id_curr_test)

print(f"Processed test data shape: {X_test_processed_df.shape}")
display(X_test_processed_df.head())


# ── 12. Visualise key features in processed test data ────────
sns.set_style("whitegrid")

numerical_features_to_plot = ['AMT_CREDIT', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY']
plt.figure(figsize=(16, 10))
for i, col in enumerate(numerical_features_to_plot):
    if col in X_test_processed_df.columns:
        plt.subplot(2, 2, i + 1)
        sns.histplot(X_test_processed_df[col], kde=True, bins=50, color='skyblue')
        plt.title(f'Distribution of {col} (Processed Test Data)')
        plt.xlabel(col)
        plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

categorical_features_to_plot = [
    'NAME_CONTRACT_TYPE_Cash loans',
    'CODE_GENDER_F',
    'FLAG_OWN_CAR_Y',
    'NAME_INCOME_TYPE_Working',
]
plt.figure(figsize=(16, 10))
for i, col in enumerate(categorical_features_to_plot):
    if col in X_test_processed_df.columns:
        plt.subplot(2, 2, i + 1)
        sns.countplot(
            x=X_test_processed_df[col],
            hue=X_test_processed_df[col],
            palette='viridis',
            legend=False,
        )
        plt.title(f'Count of {col} (Processed Test Data)')
        plt.xlabel(col)
        plt.ylabel('Count')
plt.tight_layout()
plt.show()

print("All preprocessing complete. X_processed_df and X_test_processed_df are ready for modelling.")
